# Ethiopia Financial Inclusion — Data Exploration, Enrichment & EDA
### Tasks 1 & 2 | Selam Analytics

This notebook covers:
- **Task 1:** Loading the unified dataset, understanding its schema, exploring record composition, and documenting enrichment additions.
- **Task 2:** Full exploratory data analysis — dataset overview, Access trends, Usage trends, event timeline, key insights, and data limitations.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")
plt.rcParams["figure.facecolor"] = "white"

df = pd.read_csv("../data/raw/ethiopia_fi_unified_data.csv", parse_dates=["observation_date"])
ref = pd.read_csv("../data/raw/reference_codes.csv")

print(f"Unified dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Reference codes: {ref.shape[0]} rows")
df.head(3)

Unified dataset: 62 rows, 35 columns
Reference codes: 71 rows


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes,parent_id
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,...,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Baseline year,NaN,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,...,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,...,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,NaN,NaN,NaN


## Part 1: Schema Understanding

The unified schema stores four kinds of records in a single flat table, distinguished by `record_type`:

In [2]:
print(df["record_type"].value_counts())
print()
for rt in ["observation", "event", "impact_link", "target"]:
    desc = ref[(ref["field"] == "record_type") & (ref["code"] == rt)]["description"].values
    print(f"{rt}: {desc[0] if len(desc) else ''}")

record_type
observation    35
impact_link    14
event          10
target          3
Name: count, dtype: int64

observation: Actual measured value from a source
event: Policy launch market event or milestone
impact_link: Relationship between event and indicator (links via parent_id)
target: Policy target or official goal


**How `impact_link` connects events to indicators:** each `impact_link` row has a `parent_id` pointing back to the `event` it originated from, and a `related_indicator` pointing to the `indicator_code` it affects. This lets us trace: *event → estimated impact → which indicator, by how much, over what lag.*

Below, we resolve one impact_link end-to-end as a worked example.

In [3]:
example_link = df[df["record_type"] == "impact_link"].iloc[0]
parent_event = df[df["record_id"] == example_link["parent_id"]].iloc[0]

print(f"Impact link: {example_link['record_id']}")
print(f"  Caused by event ({example_link['parent_id']}): {parent_event['indicator']}  [{parent_event['observation_date'].date()}]")
print(f"  Affects indicator: {example_link['related_indicator']}")
print(f"  Relationship: {example_link['relationship_type']}, {example_link['impact_direction']}, magnitude={example_link['impact_magnitude']}")
print(f"  Estimated impact: {example_link['impact_estimate']} over {example_link['lag_months']} months")
print(f"  Evidence basis: {example_link['evidence_basis']} (comparable country: {example_link['comparable_country']})")

Impact link: IMP_0001
  Caused by event (EVT_0001): Telebirr Launch  [2021-05-17]
  Affects indicator: ACC_OWNERSHIP
  Relationship: direct, increase, magnitude=high
  Estimated impact: 15.0 over 12.0 months
  Evidence basis: literature (comparable country: Kenya)


**Note on pillar field convention:** `pillar` is populated for `observation` and `target` records (e.g. ACCESS, USAGE, GENDER) but left empty for `event` records, since an event itself doesn't belong to a single pillar — its *impact_links* do (via `related_indicator`).

In [4]:
print("Pillar is set for observations/targets:")
print(df[df["record_type"].isin(["observation", "target"])]["pillar"].value_counts())
print()
print("Pillar is empty for events (as expected):", df[df["record_type"] == "event"]["pillar"].isna().all() or (df[df["record_type"] == "event"]["pillar"] == "").all())

Pillar is set for observations/targets:
pillar
ACCESS           19
USAGE            12
GENDER            6
AFFORDABILITY     1
Name: count, dtype: int64

Pillar is empty for events (as expected): True


## Part 2: Data Exploration

In [5]:
print("=== Records by record_type ===")
print(df["record_type"].value_counts(), "\n")

print("=== Records by pillar (observations & targets only) ===")
print(df[df["record_type"].isin(["observation", "target"])]["pillar"].value_counts(), "\n")

print("=== Records by source_type ===")
print(df["source_type"].value_counts(), "\n")

print("=== Records by confidence ===")
print(df["confidence"].value_counts())

=== Records by record_type ===
record_type
observation    35
impact_link    14
event          10
target          3
Name: count, dtype: int64 

=== Records by pillar (observations & targets only) ===
pillar
ACCESS           19
USAGE            12
GENDER            6
AFFORDABILITY     1
Name: count, dtype: int64 

=== Records by source_type ===
source_type
operator      15
survey        10
research       9
regulator      7
policy         3
calculated     2
news           2
Name: count, dtype: int64 

=== Records by confidence ===
confidence
high      45
medium    17
Name: count, dtype: int64


In [6]:
temporal = df.dropna(subset=["observation_date"])
print(f"Temporal range: {temporal['observation_date'].min().date()} to {temporal['observation_date'].max().date()}")
print(f"Span: {(temporal['observation_date'].max() - temporal['observation_date'].min()).days / 365.25:.1f} years")
print()
n_indicators = df["indicator_code"].nunique()
print(f"Unique indicator codes covered: {n_indicators}")
print(sorted(df["indicator_code"].dropna().unique()))

Temporal range: 2014-12-31 to 2030-12-31
Span: 16.0 years

Unique indicator codes covered: 31
['ACC_4G_COV', 'ACC_FAYDA', 'ACC_MM_ACCOUNT', 'ACC_MOBILE_OWN', 'ACC_MOBILE_PEN', 'ACC_OWNERSHIP', 'AFF_DATA_INCOME', 'EVT_CROSSOVER', 'EVT_ETHIOPAY', 'EVT_FAYDA', 'EVT_FX_REFORM', 'EVT_MPESA', 'EVT_MPESA_INTEROP', 'EVT_NFIS2', 'EVT_SAFARICOM', 'EVT_SAFCOM_PRICE', 'EVT_TELEBIRR', 'GEN_GAP_ACC', 'GEN_GAP_MOBILE', 'GEN_MM_SHARE', 'USG_ACTIVE_RATE', 'USG_ATM_COUNT', 'USG_ATM_VALUE', 'USG_CROSSOVER', 'USG_DIGITAL_PAY', 'USG_MPESA_ACTIVE', 'USG_MPESA_USERS', 'USG_P2P_COUNT', 'USG_P2P_VALUE', 'USG_TELEBIRR_USERS', 'USG_TELEBIRR_VALUE']


## Part 3: Dataset Overview (Task 2)

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

df["record_type"].value_counts().plot(kind="bar", ax=axes[0], color="#2980b9")
axes[0].set_title("Records by Type", fontweight="bold")
axes[0].tick_params(axis="x", rotation=30)

df[df["record_type"].isin(["observation","target"])]["pillar"].value_counts().plot(kind="bar", ax=axes[1], color="#27ae60")
axes[1].set_title("Observations/Targets by Pillar", fontweight="bold")
axes[1].tick_params(axis="x", rotation=45)

df["source_type"].value_counts().plot(kind="bar", ax=axes[2], color="#e67e22")
axes[2].set_title("Records by Source Type", fontweight="bold")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("../data/processed/dataset_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 1: dataset overview saved")

Figure 1: dataset overview saved


In [8]:
fig, ax = plt.subplots(figsize=(7, 5))
conf_order = ["high", "medium", "low"]
conf_counts = df["confidence"].value_counts().reindex(conf_order).fillna(0)
colors = ["#27ae60", "#e67e22", "#e74c3c"]
ax.bar(conf_counts.index, conf_counts.values, color=colors)
ax.set_title("Data Quality: Confidence Level Distribution", fontweight="bold")
ax.set_ylabel("Number of Records")
for i, v in enumerate(conf_counts.values):
    ax.text(i, v + 0.5, str(int(v)), ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("../data/processed/confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 2: confidence distribution saved")
print(f"\n{(conf_counts['high']/conf_counts.sum()*100):.0f}% of records are high-confidence")

Figure 2: confidence distribution saved

73% of records are high-confidence


In [9]:
obs = df[df["record_type"] == "observation"].dropna(subset=["observation_date"])
indicator_years = obs.groupby("indicator")["observation_date"].agg(["min", "max", "count"]).sort_values("min")

fig, ax = plt.subplots(figsize=(12, 8))
for i, (ind, row) in enumerate(indicator_years.iterrows()):
    ax.plot([row["min"], row["max"]], [i, i], color="#2980b9", linewidth=2)
    ax.scatter([row["min"], row["max"]], [i, i], color="#2980b9", s=20, zorder=5)
ax.set_yticks(range(len(indicator_years)))
ax.set_yticklabels(indicator_years.index, fontsize=8)
ax.set_title("Temporal Coverage by Indicator", fontweight="bold")
ax.set_xlabel("Date")
plt.tight_layout()
plt.savefig("../data/processed/temporal_coverage.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 3: temporal coverage by indicator saved")

Figure 3: temporal coverage by indicator saved


## Part 4: Access Analysis

In [10]:
acc = df[(df["indicator_code"] == "ACC_OWNERSHIP") & (df["gender"] == "all") & (df["record_type"] == "observation")].sort_values("observation_date")
print(acc[["observation_date", "value_numeric", "source_name", "confidence"]])

  observation_date  value_numeric         source_name confidence
0       2014-12-31           22.0  Global Findex 2014       high
1       2017-12-31           35.0  Global Findex 2017       high
2       2021-12-31           46.0  Global Findex 2021       high
5       2024-11-29           49.0  Global Findex 2024       high


In [11]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(acc["observation_date"], acc["value_numeric"], marker="o", markersize=8, color="#2980b9", linewidth=2)
for _, row in acc.iterrows():
    ax.annotate(f"{row['value_numeric']:.0f}%", (row["observation_date"], row["value_numeric"]),
                textcoords="offset points", xytext=(0, 10), ha="center", fontweight="bold")
ax.set_title("Ethiopia Account Ownership Trajectory (2014-2024)", fontweight="bold", fontsize=13)
ax.set_ylabel("Account Ownership (%)")
ax.set_ylim(0, 60)
plt.tight_layout()
plt.savefig("../data/processed/access_trajectory.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 4: account ownership trajectory saved")

Figure 4: account ownership trajectory saved


In [12]:
acc = acc.reset_index(drop=True)
acc["years_since_prev"] = acc["observation_date"].diff().dt.days / 365.25
acc["pp_change"] = acc["value_numeric"].diff()
acc["pp_per_year"] = acc["pp_change"] / acc["years_since_prev"]
print(acc[["observation_date", "value_numeric", "pp_change", "years_since_prev", "pp_per_year"]].round(2))

  observation_date  value_numeric  pp_change  years_since_prev  pp_per_year
0       2014-12-31           22.0        NaN               NaN          NaN
1       2017-12-31           35.0       13.0              3.00         4.33
2       2021-12-31           46.0       11.0              4.00         2.75
3       2024-11-29           49.0        3.0              2.91         1.03


**The 2021-2024 slowdown:** account ownership rose only **+3 percentage points** (46% -> 49%) between the 2021 and 2024 Findex waves — a rate of roughly **1.03 pp/year**, versus **2.75 pp/year** between 2017 and 2021, despite this being exactly the period when Telebirr (2021), M-Pesa (2023), and the Fayda digital ID rollout (2024) all launched. This is a genuinely important, slightly counter-intuitive finding: major mobile money launches did not translate into a proportionate acceleration in *overall* account ownership growth.

Plausible explanations worth flagging (not fully resolved by this dataset alone):
- Mobile money adoption may be substituting for, rather than adding to, traditional bank account growth for many users.
- The `ACC_MM_ACCOUNT` indicator (4.7% -> 9.45%, 2021-2024) shows mobile money nearly doubled in relative terms but from a small base — not yet large enough to move the aggregate account-ownership number substantially.
- The 2024 gender gap remains large (REC_0037: 14.91pp), suggesting growth is concentrated among a subset of the population rather than broad-based.

## Part 5: Usage Analysis

In [13]:
mm = df[(df["indicator_code"] == "ACC_MM_ACCOUNT") & (df["record_type"] == "observation")].sort_values("observation_date")
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mm["observation_date"], mm["value_numeric"], marker="o", markersize=8, color="#27ae60", linewidth=2)
for _, row in mm.iterrows():
    ax.annotate(f"{row['value_numeric']:.2f}%", (row["observation_date"], row["value_numeric"]),
                textcoords="offset points", xytext=(0, 10), ha="center", fontweight="bold")
ax.set_title("Mobile Money Account Ownership Rate (2021-2024)", fontweight="bold")
ax.set_ylabel("Mobile Money Account Ownership (%)")
plt.tight_layout()
plt.savefig("../data/processed/mobile_money_trend.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 5: mobile money trend saved")
print(f"\nMobile money account ownership roughly doubled: {mm['value_numeric'].iloc[0]:.2f}% -> {mm['value_numeric'].iloc[-1]:.2f}%")

Figure 5: mobile money trend saved

Mobile money account ownership roughly doubled: 4.70% -> 9.45%


In [14]:
digital_indicators = ["USG_TELEBIRR_USERS", "USG_MPESA_USERS", "USG_P2P_COUNT", "USG_DIGITAL_PAY"]
digital = df[df["indicator_code"].isin(digital_indicators) & (df["record_type"] == "observation")]
print(digital[["indicator", "value_numeric", "unit", "observation_date", "confidence"]].sort_values("observation_date").to_string(index=False))

                    indicator  value_numeric         unit observation_date confidence
        P2P Transaction Count     49700000.0 transactions       2024-07-07       high
      M-Pesa Registered Users     10800000.0        users       2024-12-31       high
Digital Payment Adoption Rate           16.0            %       2024-12-31     medium
    Telebirr Registered Users     54840000.0        users       2025-06-30       high
        P2P Transaction Count    128300000.0 transactions       2025-07-07       high


**Registered vs. active user gap:** Telebirr reports 54.84M *registered* users, but the dataset also captures M-Pesa's more transparent breakdown: **10.8M registered vs. 7.1M active (66% activity rate)**. This 34-point gap between registration and active use is an important usage-quality signal — headline "registered user" counts likely overstate genuine adoption breadth across the sector, not just for M-Pesa specifically.

In [15]:
fig, ax = plt.subplots(figsize=(6, 5))
labels = ["Registered", "90-Day Active"]
values = [10800000, 7100000]
ax.bar(labels, values, color=["#95a5a6", "#27ae60"])
for i, v in enumerate(values):
    ax.text(i, v + 100000, f"{v/1e6:.1f}M", ha="center", fontweight="bold")
ax.set_title("M-Pesa: Registered vs. Active Users (2024)", fontweight="bold")
ax.set_ylabel("Users")
plt.tight_layout()
plt.savefig("../data/processed/registered_vs_active.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 6: registered vs active gap saved")

Figure 6: registered vs active gap saved


## Part 6: Event Timeline

In [16]:
events = df[df["record_type"] == "event"].sort_values("observation_date")

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(acc["observation_date"], acc["value_numeric"], marker="o", color="#2980b9", linewidth=2, label="Account Ownership (%)", zorder=3)

for _, ev in events.iterrows():
    ax.axvline(ev["observation_date"], color="#e67e22", alpha=0.5, linewidth=1)
    ax.text(ev["observation_date"], 57, ev["indicator"], rotation=90, fontsize=7, va="top", color="#a04000")

ax.set_title("Account Ownership Trend with Cataloged Events Overlaid", fontweight="bold", fontsize=13)
ax.set_ylabel("Account Ownership (%)")
ax.set_ylim(0, 62)
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig("../data/processed/event_timeline.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 7: event timeline saved")
print(f"\n{len(events)} events cataloged, {events['observation_date'].min().date()} to {events['observation_date'].max().date()}")

Figure 7: event timeline saved

10 events cataloged, 2021-05-17 to 2025-12-18


## Part 7: Key Insights

1. **Account ownership growth has slowed markedly (2021-2024)**, rising only +3pp (46% -> 49%) despite Telebirr, M-Pesa, and Fayda digital ID all launching in this window — a much slower pace than the +11pp seen 2017-2021. Mobile money expansion has not yet translated into proportionate overall account growth.

2. **Mobile money is growing fast but from a small base**: mobile-money-specific account ownership nearly doubled (4.7% -> 9.45%, 2021-2024), showing real momentum, but it's still a small share of the 49% overall account ownership figure — most accounts remain traditional bank accounts.

3. **A large registered-vs-active gap exists in usage data**: M-Pesa shows only 66% of registered users (7.1M of 10.8M) are 90-day active, suggesting registration counts across the sector likely overstate genuine engagement.

4. **The gender gap remains substantial and is a real barrier, not a rounding error**: precisely-sourced 2024 data (added via enrichment) shows a 14.91pp gap (56.53% male vs. 41.62% female) in account ownership, and women hold only 14% of mobile money accounts specifically (REC_0029) — far from the NBE's 2030 parity target of 50%.

5. **P2P digital transactions overtook ATM transactions for the first time in FY2024/25** (EVT_0006, Oct 2024), with a P2P/ATM crossover ratio of 1.08 — a genuine milestone suggesting a structural shift from cash-adjacent behavior (ATM withdrawal) toward direct digital payments.

6. **Infrastructure investment appears linked to the 2025 4G coverage jump**: population 4G coverage doubled from 37.5% (FY2022/23) to 70.8% (FY2024/25) — a plausible enabling factor for the digital payment growth seen in the same window, though this dataset can only show correlation in time, not proven causation (see Limitations).

## Part 8: Data Limitations

- **Correlation in time is not causation.** Every `impact_link` record in this dataset is an *estimate* (`evidence_basis`: literature/empirical/theoretical, often drawing on comparable-country precedents like Kenya or India) of how much an event plausibly moved an indicator. These are analyst judgments informed by evidence, not proven causal effects — multiple events often overlap in time (e.g., Telebirr launch, NFIS-II strategy, and organic growth all occurred in the same 2021-2022 window), making it impossible to cleanly separate their individual contributions from this data alone.

- **Sparse historical coverage for core Access indicators.** Account ownership has only 5 survey-wave data points across 10 years (2014, 2017, 2021, 2021-gender-split, 2024) since Global Findex surveys run roughly every 3 years. Trend and growth-rate analysis is therefore based on a small number of anchor points, not continuous data — the "2021-2024 slowdown" finding, while genuine, rests on comparing just two survey waves.

- **No verified 2011 baseline exists.** Several secondary sources reference "22% account ownership in 2011" for Ethiopia; we traced this to a World Bank primary source confirming it is actually the **2014** figure, and confirmed Ethiopia was not covered in the 2011 Global Findex wave at all. We deliberately did not fabricate a 2011 data point (see `data_enrichment_log.md`).

- **Confidence levels are analyst judgments, not statistical measures.** The `confidence` field (high/medium/low) reflects source reliability as assessed by the data collector, not a formal statistical confidence interval — different collectors could reasonably rate the same source differently.

- **Regional and sub-national detail is essentially absent.** Nearly all records have `location = national` with an empty `region` field; the dataset cannot currently support any urban/rural or region-specific analysis, despite this being a first-order driver of financial inclusion in Ethiopia per the broader literature.

- **Enrichment sources introduce their own uncertainty.** The 5 new records added in this round (REC_0034-0038) are all sourced from a single secondary academic paper rather than the primary Global Findex microdata directly, which is why they're marked `medium` confidence rather than `high` (except the directly-computed gender gap, REC_0037).